# 03 — Primeiro baseline de RUL no FD001

Neste notebook será construído o primeiro modelo de referência para estimar a vida útil remanescente dos motores. O objetivo ainda não é obter o melhor resultado possível, mas estabelecer uma comparação reproduzível antes de testar modelos mais complexos.

A avaliação será feita por motor. Em termos simples, alguns motores serão usados para treinamento e motores inteiros, que não apareceram nessa etapa, serão usados para validação. Essa separação reduz o risco de o modelo aprender características específicas de uma trajetória e parecer melhor do que realmente é.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'vscode'

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import (
    add_rul_label, evaluate_rul, make_random_forest_regressor,
    median_rul_prediction, nonconstant_features, split_by_unit,
)
from multilang import LangMap

lang = LangMap(source='pt', target='pt')
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Construção do RUL

O RUL será calculado como a diferença entre o último ciclo observado de um motor e o ciclo atual. Essa regra transforma cada linha da trajetória em uma observação com uma resposta conhecida para o treinamento. Ela é adequada ao conjunto de treino, no qual a trajetória completa até a falha está disponível.

In [2]:
fd001 = load_split(DATA_DIR, 'train', 'FD001')
fd001_labeled = add_rul_label(fd001)
rul_summary = fd001_labeled['rul'].describe().rename({
    'count': 'observações',
    'mean': 'média',
    'std': 'desvio padrão',
    'min': 'mínimo',
    '25%': 'primeiro quartil',
    '50%': 'mediana',
    '75%': 'terceiro quartil',
    'max': 'máximo',
}).to_frame('RUL em ciclos')
rul_summary

,RUL em ciclos
observações,20631.000000
média,107.807862
desvio padrão,68.880990
mínimo,0.000000
primeiro quartil,51.000000
mediana,103.000000
terceiro quartil,155.000000
máximo,361.000000


**Insights:** O RUL variou de 0 a 361 ciclos, com mediana de 103 ciclos nas 20.631 observações do FD001. Essa distribuição reúne muitos ciclos de motores em diferentes momentos de suas vidas, por isso não deve ser interpretada como a distribuição de 100 motores independentes. A variável será usada como resposta do modelo, enquanto os sensores, as configurações operacionais e o ciclo serão usados como informações disponíveis no momento da previsão.

## 2. Separação por motor

A divisão será feita com grupos formados pelo identificador do motor, usando `GroupShuffleSplit` do scikit-learn (Pedregosa et al., 2011). Em linguagem simples, nenhuma trajetória será quebrada entre treinamento e validação. Essa escolha é mais adequada ao problema do que uma divisão aleatória de linhas, porque ciclos vizinhos do mesmo motor carregam informações muito semelhantes.

In [3]:
train_df, validation_df = split_by_unit(fd001_labeled, test_size=0.2, random_state=42)
split_summary = pd.DataFrame({
    'conjunto': ['treinamento', 'validação'],
    'motores': [train_df['unit'].nunique(), validation_df['unit'].nunique()],
    'observações': [len(train_df), len(validation_df)],
    'primeiro_motor': [train_df['unit'].min(), validation_df['unit'].min()],
    'último_motor': [train_df['unit'].max(), validation_df['unit'].max()],
})
split_summary

,conjunto,motores,observações,primeiro_motor,último_motor
0,treinamento,80,16561,2,100
1,validação,20,4070,1,91


**Insights:** A validação foi formada por 20 motores inteiros, enquanto 80 motores foram usados no treinamento. Isso produz uma situação mais próxima do uso real: o modelo recebe uma trajetória de um motor que não conheceu durante o treinamento. O resultado continuará sendo uma estimativa de desempenho, porque uma única divisão de motores não substitui uma validação cruzada futura.

## 3. Referência ingênua

Antes do modelo preditivo, será usada uma referência simples: o RUL mediano observado nos motores de treinamento será repetido para todos os ciclos de validação. Essa referência não aprende o comportamento dos sensores, mas indica o erro que precisará ser superado para que um modelo seja considerado útil.

In [5]:
y_train = train_df['rul']
y_validation = validation_df['rul']
naive_prediction = median_rul_prediction(y_train, len(validation_df))
naive_metrics = pd.DataFrame([evaluate_rul(y_validation, naive_prediction)])
naive_metrics.insert(0, 'modelo', 'Mediana do treinamento')
naive_metrics

,modelo,mae_ciclos,rmse_ciclos
0,Mediana do treinamento,55.127518,65.699059


**Insights:** A referência constante apresentou MAE de 55,1 ciclos e RMSE de 65,7 ciclos. Esses valores representam o ponto de partida que deverá ser superado pelo modelo que utiliza os sensores. O erro médio absoluto é lido em ciclos: neste caso, a previsão ficou, em média, cerca de 55 ciclos distante do RUL observado.

## 4. Primeiro modelo preditivo

Será utilizado um Random Forest para regressão (Breiman, 2001). O método combina várias árvores de decisão, de modo que diferentes regras aprendidas a partir dos sensores sejam agregadas em uma única previsão. A técnica foi escolhida como primeiro modelo por conseguir representar relações não lineares e por não exigir que todas as variáveis estejam na mesma escala.

In [6]:
candidate_features = ['cycle', 'setting1', 'setting2', 'setting3', *[f's{i}' for i in range(1, 22)]]
features = nonconstant_features(train_df, candidate_features)

model = make_random_forest_regressor(random_state=42)
model.fit(train_df[features], y_train)
model_prediction = model.predict(validation_df[features])
model_metrics = pd.DataFrame([evaluate_rul(y_validation, model_prediction)])
model_metrics.insert(0, 'modelo', 'Random Forest')
metrics = pd.concat([naive_metrics, model_metrics], ignore_index=True)
metrics

,modelo,mae_ciclos,rmse_ciclos
0,Mediana do treinamento,55.127518,65.699059
1,Random Forest,23.832730,31.394413


**Insights:** O Random Forest apresentou MAE de 23,8 ciclos e RMSE de 31,4 ciclos, melhorando a referência constante em aproximadamente 31 ciclos no erro médio absoluto. Esse resultado sugere que os sensores e o ciclo acrescentaram informação preditiva no FD001. Ainda não deverá ser tratado como desempenho final, pois foi obtido em uma única divisão do conjunto e não considera alertas antecipados, custos de erro ou intervalos de incerteza. A escolha de retirar colunas constantes foi feita apenas com os motores de treinamento, preservando a separação metodológica.

In [7]:
predictions = validation_df[['unit', 'cycle', 'rul']].copy()
predictions['rul_predito'] = model_prediction
fig = px.scatter(
    predictions,
    x='rul',
    y='rul_predito',
    labels={'rul': 'RUL observado (ciclos)', 'rul_predito': 'RUL previsto (ciclos)'},
    title=lang({'title': 'RUL observado e previsto — validação por motor'} )['title'],
    height=600,
)
max_rul = max(predictions['rul'].max(), predictions['rul_predito'].max())
fig.add_shape(type='line', x0=0, y0=0, x1=max_rul, y1=max_rul,
              line={'color': 'gray', 'dash': 'dash'})
fig.show()

**Insights:** Quanto mais próximos os pontos estiverem da linha diagonal, mais próxima estará a previsão do RUL observado. Pontos acima da linha representam uma vida restante superestimada; pontos abaixo representam uma previsão menor que o observado. Esse gráfico ajuda a enxergar o comportamento dos erros, mas não informa sozinho se o modelo é adequado para decidir o momento de uma manutenção.

## Conclusão desta etapa

Foi estabelecido um primeiro ponto de comparação para a previsão de RUL no FD001. A referência ingênua informa o erro de uma estratégia sem sensores, e o Random Forest permite verificar se os sinais acrescentam capacidade preditiva quando a avaliação é feita em motores não utilizados no treinamento.

Ainda não será concluído que o modelo está pronto para uso operacional. As próximas etapas deverão repetir a avaliação em diferentes divisões por motor, testar outros conjuntos de condições operacionais e acrescentar early warning, custos de falso alarme e incerteza.

## Referências

Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32.

Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.